In [19]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer, OneHotEncoder
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.sql.functions import skewness, kurtosis, stddev, mean, expr
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os
import psutil
import builtins

total_ram_gb = psutil.virtual_memory().total / 1e9
print(f"Total system RAM: {total_ram_gb:.1f} GB")
driver_mem_gb = builtins.max(4, builtins.min(10, int(total_ram_gb * 0.5)))
print(f"Setting spark.driver.memory to {driver_mem_gb}g")

spark = (
    SparkSession.builder
    .appName("BusReliabilityYorkshire")
    .master("local[4]")                                           
    .config("spark.driver.memory", f"{driver_mem_gb}g")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.shuffle.partitions", "32")                  
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.memory.fraction", "0.8")
    .config("spark.memory.storageFraction", "0.3")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark Session created: {spark.version}")
print("Spark UI:           ", spark.sparkContext.uiWebUrl, " <-- screenshot this while a job runs")


Total system RAM: 17.2 GB
Setting spark.driver.memory to 8g
Spark Session created: 4.1.1
Spark UI:            http://192.168.111.15:4040  <-- screenshot this while a job runs


In [20]:
def explore_dataset(df, name):
    print(f"\n{'='*50}")
    print(f"Exploring: {name}")
    print(f"{'='*50}")
    count = df.count()
    print(f"Total records: {count:,}")
    print("\nSchema:")
    df.printSchema()
    print("\nSample data (5 rows):")
    df.show(5, truncate=False)
    print("\nNull values per column:")
    null_counts = df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns])
    null_counts.show()
    numeric_cols = [c for c, t in df.dtypes if t in ("int", "double", "float", "long")]
    if numeric_cols:
        print("\nSummary statistics for numeric columns:")
        df.select(numeric_cols).describe().show()

In [21]:
# ============================================
# 2. DATA LOADING
# ============================================
def load_gtfs_data(base_path):
    """
    Load all GTFS files from the specified directory
    """
    print("Loading GTFS files...")
 
    # Load each GTFS file (short name -> path)
    files = {
        'agency': f"/Users/paridhi/Downloads/itm_yorkshire_gtfs/agency.txt",
        'calendar': f"/Users/paridhi/Downloads/itm_yorkshire_gtfs/calendar.txt",
        'calendar_dates': f"/Users/paridhi/Downloads/itm_yorkshire_gtfs/calendar_dates.txt",
        'routes': f"/Users/paridhi/Downloads/itm_yorkshire_gtfs/routes.txt",
        'shapes': f"/Users/paridhi/Downloads/itm_yorkshire_gtfs/shapes.txt",
        'stop_times': f"/Users/paridhi/Downloads/itm_yorkshire_gtfs/stop_times.txt",
        'stops': f"/Users/paridhi/Downloads/itm_yorkshire_gtfs/stops.txt",
        'trips': f"/Users/paridhi/Downloads/itm_yorkshire_gtfs/trips.txt",
    }
 
    # Load dataframes with schema inference
    df_agency = spark.read.option("header", "true") \
                         .option("inferSchema", "true") \
                         .csv(files["agency"])
 
    df_calendar = spark.read.option("header", "true") \
                           .option("inferSchema", "true") \
                           .csv(files["calendar"])
 
    df_calendar_dates = spark.read.option("header", "true") \
                                  .option("inferSchema", "true") \
                                  .csv(files["calendar_dates"])
 
    df_routes = spark.read.option("header", "true") \
                         .option("inferSchema", "true") \
                         .csv(files["routes"])
 
    df_shapes = spark.read.option("header", "true") \
                         .option("inferSchema", "true") \
                         .csv(files["shapes"])
 
    df_stop_times = spark.read.option("header", "true") \
                              .option("inferSchema", "true") \
                              .csv(files["stop_times"])
 
    df_stops = spark.read.option("header", "true") \
                        .option("inferSchema", "true") \
                        .csv(files["stops"])
 
    df_trips = spark.read.option("header", "true") \
                        .option("inferSchema", "true") \
                        .csv(files["trips"])
 
    print("All GTFS files loaded successfully!")
    print(f"Stop Times records: {df_stop_times.count():,}")
    print(f"Trips records: {df_trips.count():,}")
    print(f"Routes records: {df_routes.count():,}")
    print(f"Stops records: {df_stops.count():,}")
 
    return {
        'agency': df_agency,
        'calendar': df_calendar,
        'calendar_dates': df_calendar_dates,
        'routes': df_routes,
        'shapes': df_shapes,
        'stop_times': df_stop_times,
        'stops': df_stops,
        'trips': df_trips,
    }
 
 
# Load the data
base_path = "/Users/paridhi/Downloads/itm_yorkshire_gtfs"  
gtfs_data = load_gtfs_data(base_path)  
 
for name, df in gtfs_data.items():
    explore_dataset(df, name)

Loading GTFS files...


All GTFS files loaded successfully!


Stop Times records: 6,481,836
Trips records: 138,436
Routes records: 1,243
Stops records: 32,583

Exploring: agency
Total records: 57

Schema:
root
 |-- agency_id: string (nullable = true)
 |-- agency_name: string (nullable = true)
 |-- agency_url: string (nullable = true)
 |-- agency_timezone: string (nullable = true)
 |-- agency_lang: string (nullable = true)
 |-- agency_phone: string (nullable = true)
 |-- agency_noc: string (nullable = true)


Sample data (5 rows):
+---------+---------------------------+--------------------------+---------------+-----------+------------+----------+
|agency_id|agency_name                |agency_url                |agency_timezone|agency_lang|agency_phone|agency_noc|
+---------+---------------------------+--------------------------+---------------+-----------+------------+----------+
|OP10924  |Bee Network                |https://www.traveline.info|Europe/London  |EN         |NULL        |BNSM      |
|OP1123   |South Yorkshire Future Tram|https://www

+--------+------------+------------+-----------------+-------------------+
|shape_id|shape_pt_lat|shape_pt_lon|shape_pt_sequence|shape_dist_traveled|
+--------+------------+------------+-----------------+-------------------+
|       0|           0|           0|                0|            6068861|
+--------+------------+------------+-----------------+-------------------+


Summary statistics for numeric columns:


+-------+-------------------+-------------------+-----------------+
|summary|       shape_pt_lat|       shape_pt_lon|shape_pt_sequence|
+-------+-------------------+-------------------+-----------------+
|  count|            6068861|            6068861|          6068861|
|   mean| 53.662067957660845|-1.2776909216519663|740.4427570511172|
| stddev|0.25179669193759374|0.46442970341250017|584.3709936290214|
|    min|           53.13465|           -2.74762|                0|
|    max|         54.5452786|        0.145142955|             4752|
+-------+-------------------+-------------------+-----------------+


Exploring: stop_times


Total records: 6,481,836

Schema:
root
 |-- trip_id: string (nullable = true)
 |-- arrival_time: string (nullable = true)
 |-- departure_time: string (nullable = true)
 |-- stop_id: string (nullable = true)
 |-- stop_sequence: integer (nullable = true)
 |-- stop_headsign: string (nullable = true)
 |-- pickup_type: integer (nullable = true)
 |-- drop_off_type: integer (nullable = true)
 |-- shape_dist_traveled: string (nullable = true)
 |-- timepoint: integer (nullable = true)


Sample data (5 rows):
+------------------------------------------+------------+--------------+------------+-------------+-------------+-----------+-------------+-------------------+---------+
|trip_id                                   |arrival_time|departure_time|stop_id     |stop_sequence|stop_headsign|pickup_type|drop_off_type|shape_dist_traveled|timepoint|
+------------------------------------------+------------+--------------+------------+-------------+-------------+-----------+-------------+----------------

+-------+------------+--------------+-------+-------------+-------------+-----------+-------------+-------------------+---------+
|trip_id|arrival_time|departure_time|stop_id|stop_sequence|stop_headsign|pickup_type|drop_off_type|shape_dist_traveled|timepoint|
+-------+------------+--------------+-------+-------------+-------------+-----------+-------------+-------------------+---------+
|      0|           0|             0|      0|            0|      6481836|          0|            0|            6481836|        0|
+-------+------------+--------------+-------+-------------+-------------+-----------+-------------+-------------------+---------+


Summary statistics for numeric columns:


+-------+------------------+--------------------+--------------------+-------------------+
|summary|     stop_sequence|         pickup_type|       drop_off_type|          timepoint|
+-------+------------------+--------------------+--------------------+-------------------+
|  count|           6481836|             6481836|             6481836|            6481836|
|   mean|29.028599304271197|0.011133728159737457|0.010640966540961543|0.13167519202892514|
| stddev|21.937954936473187| 0.10492745091518997| 0.10260476595278781| 0.3381373293096347|
|    min|                 0|                   0|                   0|                  0|
|    max|               180|                   1|                   1|                  1|
+-------+------------------+--------------------+--------------------+-------------------+


Exploring: stops
Total records: 32,583

Schema:
root
 |-- stop_id: string (nullable = true)
 |-- stop_code: string (nullable = true)
 |-- stop_name: string (nullable = true)
 |-- 